In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# 1. Cargamos las variables de entorno desde tu archivo .env
load_dotenv()

# 2. Recuperamos las credenciales correctas de manera segura
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

# 3. Validación rápida en la consola del cuaderno
if not GROQ_API_KEY:
    print("⚠️ ¡Alerta! No se encontró la GROQ_API_KEY en tu archivo .env")
else:
    print("✅ API Keys de Groq y Tavily cargadas correctamente.")

# 4. Inicializamos el modelo activo de Groq (Llama 3.3) para esta clase
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)

In [ ]:
profile = {
    "name": "Sarah",
    "full_name": "Sarah Messi",
    "user_profile_background": "Ingeniera de software senior liderando un equipo de 5 desarrolladores"
}

# Verificación rápida en tu celda
print(f"👤 Perfil cargado: {profile['full_name']} - {profile['user_profile_background']}")

In [ ]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Newsletters de marketing, e-mails de spam, comunicados generales de la empresa",
        "notify": "Miembro del equipo convaleciente, notificaciones del sistema de build, Actualizaciones del status del proyecto",
        "respond": "Preguntas directas de miembros del equipo, solicitudes de reunión, informes de bugs críticos"
    },
    "agent_instructions": "Usa estas herramientas cuando sea apropiado para ayudar a gestionar las tareas de Sarah de forma eficiente."
}

# Validación rápida para verificar que el prompt se lee bien en el cuaderno
print("🤖 Instrucciones del Agente de Triaje preparadas.")
print(f"👉 Reglas de respuesta: {prompt_instructions['triage_rules']['respond']}")

In [ ]:
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Messi <sarah.messi@company.com>",
    "subject": "Duda rápida sobre la documentación de la API",
    "body": """Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice"""
}

# Validación para ver cómo lo recibe el entorno
print(f"📧 Correo entrante de: {email['from']}")
print(f"📋 Asunto: {email['subject']}")

In [ ]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model  # Directo desde el paquete principal

In [ ]:
import os
from langchain_groq import ChatGroq

# Inicializamos el modelo de Groq usando las variables de tu entorno
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv('GROQ_API_KEY'),
    temperature=0.0  # Temperatura en 0 para que la clasificación sea exacta
)

# Validación rápida en el cuaderno
print("🚀 Motor Llama 3.3 en Groq listo para el triaje de Sarah, messirve.")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Router(BaseModel):
    """Analiza el e-mail no leído y enrútalo de acuerdo con su contenido."""

    reasoning: str = Field(
        description="Razonamiento paso a paso que justifica la clasificación elegida."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description=(
            "La clasificación del e-mail: 'ignore' para e-mails irrelevantes, "
            "'notify' para informaciones importantes que no necesitan de una respuesta, "
            "'respond' para e-mails que requieren una respuesta"
        )
    )

# Validación rápida de la estructura
print("✅ Esquema Pydantic 'Router' validado con éxito.")

In [ ]:
llm_router = llm.with_structured_output(Router)

In [ ]:
from prompts import triage_system_prompt, triage_user_prompt

In [ ]:
system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    user_profile_background=profile["user_profile_background"],
    triage_ignore=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_respond=prompt_instructions["triage_rules"]["respond"],
    examples=""  # Pasamos un string vacío en lugar de None para no ensuciar el texto
)

# Validación rápida para ver cómo quedó el prompt final estructurado
print("📝 'system_prompt' generado con éxito. Listo para el backend.")

In [ ]:
user_prompt = triage_user_prompt.format(
    author=email["from"],
    to=email["to"],
    subject=email["subject"],
    email_thread=email["body"]
)

# Validación rápida en el cuaderno
print("📨 'user_prompt' generado con éxito. Contiene el correo de Alice.")

In [ ]:
result = llm_router.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
])

In [ ]:
# Muestra el resultado como un diccionario nativo de Python
print(result.model_dump())

In [ ]:
%pip install -qU langgraph

In [ ]:
from langchain_core.tools import tool

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Escribe y envía un e-mail."""
    # Respuesta de placeholder - en un aplicativo real, enviaría el e-mail
    return f"E-mail enviado para {to} con el asunto '{subject}'"

# Validación rápida para verificar que LangChain lo reconoce como herramienta
print(f"🔧 Herramienta registrada: {write_email.name}")
print(f"📝 Descripción: {write_email.description}")

In [ ]:
from typing import List
from langchain_core.tools import tool

@tool
def schedule_meeting(
    attendees: List[str], 
    subject: str, 
    duration_minutes: int, 
    preferred_day: str
) -> str:
    """Programa una reunión en el calendario."""
    return f"Reunión '{subject}' programada para {preferred_day} con {len(attendees)} participantes"

# Validación rápida de la herramienta
print(f"🔧 Herramienta registrada: {schedule_meeting.name}")

In [ ]:
from langchain_core.tools import tool

@tool
def check_calendar_availability(day: str) -> str:
    """Verifica la disponibilidad de agenda para una fecha determinada."""
    return f"Horarios disponibles en {day}: 9:00 AM, 2:00 PM, 4:00 PM"

# Validación rápida de la herramienta
print(f"🔧 Herramienta registrada: {check_calendar_availability.name}")

In [ ]:
from prompts import agent_system_prompt

def create_prompt(state):
    return [
        {
            "role": "system",
            "content": agent_system_prompt.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile
            )
        }
    ] + state['messages']

# Validación rápida de la estructura
print("⚙️ Función 'create_prompt' definida correctamente para LangGraph.")

In [ ]:
from prompts import agent_system_prompt

print(agent_system_prompt)

In [ ]:
from langgraph.prebuilt import create_react_agent

In [ ]:
tools = [write_email, schedule_meeting, check_calendar_availability]

# Verificamos rápidamente cuántas herramientas tenemos cargadas en la lista
print(f"📦 Total de herramientas listas para el agente: {len(tools)}")
print(f"🔧 Nombres: {[tool.name for tool in tools]}")

In [ ]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

# Inicializamos el modelo de Groq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Creamos el agente ReAct usando el parámetro 'prompt' que pide tu versión de LangGraph
agent = create_react_agent(
    model=llm,  
    tools=tools,
    prompt=create_prompt
)

print("🤖 ¡Agente ReAct inicializado con Groq usando el parámetro 'prompt' con éxito!")

In [ ]:
response = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Cuál es mi disponibilidad para el martes?"
    }]}
)

# Imprimimos el último mensaje del flujo para ver la respuesta final del agente
print(response["messages"][-1].content)

In [ ]:
from typing import TypedDict
from typing_extensions import Annotated
from langgraph.graph import add_messages

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

print("📐 Estructura 'State' definida con éxito para el Grafo.")

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

print("📊 Módulos del grafo e IPython importados con éxito.")

In [ ]:
def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
        examples=None
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    
    if result.classification == "respond":
        print("📧 Clasificación: RESPONDER - Este e-mail requiere una respuesta")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas")
        update = None
        goto = "__end__"
    elif result.classification == "notify":
        print("🔔 Clasificación: NOTIFICAR - Este e-mail contiene informaciones importantes")
        update = None
        goto = "__end__"
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")
        
    return Command(goto=goto, update=update)

print("🎯 Nodo 'triage_router' definido y tipado correctamente.")

In [ ]:
from langgraph.graph import StateGraph, START

# 1. Instanciamos el grafo pasándole la estructura del estado
email_agent = StateGraph(State)

# 2. Agregamos los nodos (sin reasignar la variable)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)

# 3. Definimos el punto de entrada
email_agent.add_edge(START, "triage_router")

# 4. Compilamos el grafo (aquí sí guardamos el resultado final)
app = email_agent.compile()

print("📊 ¡Grafo de flujo compilado y listo para producción con éxito!")

In [ ]:
# Renderizamos el diagrama de flujo del sistema multi-agente
display(Image(app.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
email_input = {
    "author": "Equipo de Marketing <marketing@amazingdeals.com>",
    "to": "Sarah Messi <sarah.chen@company.com>",
    "subject": "🔥 OFERTA EXCLUSIVA: ¡Descuento por Tiempo Limitado en Herramientas para Desarrolladores! 🔥",
    "email_thread": """Estimado(a) Desarrollador(a),

¡No pierda esta oportunidad INCREÍBLE! 🚀 POR TIEMPO LIMITADO, obtenga un 80% DE DESCUENTO en nuestro Paquete Premium para Desarrolladores. 

✨ RECURSOS:
- Autocompletado de código revolucionario con IA
- Entorno de desarrollo basado en la nube
- Soporte al cliente 24/7
- ¡Y mucho más!

💰 Precio normal: USD 999/mes
🎉 SU PRECIO ESPECIAL: ¡Solo USD 199/mes!

🕒 ¡Apúrese! Esta oferta expira en: ¡SOLO 24 HORAS!

Haga clic aquí para canjear su descuento: https://amazingdeals.com/special-offer

Atentamente,
Equipo de Marketing
---
Para cancelar la suscripción, haga clic aquí"""
}

print("📥 Objeto 'email_input' cargado correctamente con montos en USD.")

In [ ]:
response = app.invoke({"email_input": email_input})

In [ ]:
# Porque se empaco el Kernel ja ja 
def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    # MAPEOS TOTALMENTE AJUSTADOS A LA PLANTILLA DEL CURSO
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_ignore=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_respond=prompt_instructions["triage_rules"]["respond"],  # <- ¡CORREGIDO ACÁ!
        examples=None
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    
    if result.classification == "respond":
        print("📧 Clasificación: RESPONDER - Este e-mail requiere una respuesta")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas")
        update = None
        goto = "__end__"
    elif result.classification == "notify":
        print("🔔 Clasificación: NOTIFICAR - Este e-mail contiene informaciones importantes")
        update = None
        goto = "__end__"
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")
        
    return Command(goto=goto, update=update)

print("🎯 Nodo 'triage_router' sincronizado por completo con las llaves del prompt.")

In [ ]:
from langgraph.graph import StateGraph, START

# Volvemos a armar el flujo para que tome la función que acabamos de registrar
email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)
email_agent.add_edge(START, "triage_router")

# Compilamos de nuevo creando una 'app' fresca
app = email_agent.compile()

print("📊 ¡Grafo actualizado y re-compilado con la nueva función!")


In [ ]:
response = app.invoke({"email_input": email_input})

In [ ]:
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Messi <sarah.chen@company.com>",
    "subject": "Consulta rápida sobre la documentación de la API",
    "email_thread": """Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice"""
}

print("📥 Objeto 'email_input' (Consulta de API) cargado con éxito.")

In [ ]:
# Parchando errores
from langgraph.graph import StateGraph, START

# Volvemos a armar la estructura del grafo en la memoria limpia
email_agent = StateGraph(State)

# Registramos los nodos
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)

# Definimos el punto de partida
email_agent.add_edge(START, "triage_router")

# AQUÍ DEFINIMOS 'app'
app = email_agent.compile()

print("📊 Variable 'app' creada y compilada con éxito en la memoria.")

In [ ]:
# 1. Definimos la configuración de seguridad para evitar bucles infinitos
# 'recursion_limit': 10 permite que el grafo haga hasta 10 saltos entre nodos antes de frenar
configuracion = {"recursion_limit": 10}

print("🚀 Iniciando la ejecución del sistema multi-agente...\n" + "-"*50)

try:
    # 2. Invocamos al grafo compilado ('app') pasándole el correo actual
    response = app.invoke(
        {"email_input": email_input}, 
        config=configuracion
    )
    
    print("-"*50 + "\n✅ ¡El grafo terminó su ejecución con éxito!")
    
    # 3. Validamos si el flujo llegó a generar mensajes en el agente de respuesta
    if "messages" in response and len(response["messages"]) > 0:
        print("\n🤖 Respuesta generada por el agente:")
        print(response["messages"][-1].content)
    else:
        print("\n📭 El flujo terminó en el enrutador (el correo fue ignorado o notificado).")

except Exception as e:
    print("-"*50)
    print(f"🛑 El grafo se detuvo por seguridad o error de ejecución.")
    print(f"🔍 Detalle del error: {e}")

In [ ]:
from typing import TypedDict, Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.types import Command

# 1. Re-definimos el esquema del Estado (State)
class State(TypedDict):
    email_input: dict
    messages: list

print("📝 Estructura 'State' registrada.")

# 2. Re-construimos y compilamos el grafo
try:
    email_agent = StateGraph(State)
    email_agent.add_node("triage_router", triage_router)
    email_agent.add_node("response_agent", agent)
    email_agent.add_edge(START, "triage_router")
    
    app = email_agent.compile()
    print("📊 ¡Variable 'app' creada y compilada con éxito!")
except NameError as e:
    print(f"\n⚠️ Falta ejecutar alguna celda previa: {e}")
    print("👉 Asegurate de haber corrido antes las celdas donde definiste 'triage_router' y 'agent'.")

In [ ]:
from typing import Literal
from langgraph.types import Command

def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    # Formateo correcto alineado a las llaves de la plantilla
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_ignore=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_respond=prompt_instructions["triage_rules"]["respond"],
        examples=None
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    
    if result.classification == "respond":
        print("📧 Clasificación: RESPONDER - Este e-mail requiere una respuesta")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas")
        update = None
        goto = "__end__"
    elif result.classification == "notify":
        print("🔔 Clasificación: NOTIFICAR - Este e-mail contiene informaciones importantes")
        update = None
        goto = "__end__"
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")
        
    return Command(goto=goto, update=update)

print("🎯 Nodo 'triage_router' registrado en memoria.")

In [ ]:
from langgraph.graph import StateGraph, START

email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)
email_agent.add_edge(START, "triage_router")

app = email_agent.compile()
print("📊 ¡Variable 'app' creada y compilada con éxito!")

In [ ]:
import os
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START

# 1. Inicializamos el modelo de Groq de forma segura
if 'llm' not in locals():
    if 'llm_router' in locals():
        llm = llm_router
    else:
        llm = ChatGroq(model="llama-3.3-70b-specdec", temperature=0)

# 2. Lista de herramientas vacía por ahora para evitar loops masivos
tools_del_agente = [] 

# 3. Re-creamos el 'agent' usando 'prompt' en lugar de 'state_modifier'
agent = create_react_agent(
    model=llm, 
    tools=tools_del_agente,
    prompt="Sos un asistente corporativo experto. Responde de forma clara y profesional al email del usuario." # <- ¡CORREGIDO ACÁ!
)
print("🤖 Agente 'agent' re-creado con éxito con la sintaxis nueva.")

# 4. Armamos y compilamos el grafo
email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)
email_agent.add_edge(START, "triage_router")

app = email_agent.compile()
print("📊 ¡Variable 'app' creada y compilada con éxito!")

In [ ]:
configuracion = {"recursion_limit": 10}

try:
    print("🚀 Invocando al sistema con el correo de Alice...\n" + "-"*50)
    response = app.invoke({"email_input": email_input}, config=configuracion)
    print("-"*50 + "\n✅ ¡El grafo terminó su ejecución con éxito!")
    
    if "messages" in response and len(response["messages"]) > 0:
        print("\n🤖 Respuesta del agente:\n")
        print(response["messages"][-1].content)
    else:
        print("\n📭 El flujo terminó en el router (Correo ignorado/notificado).")
except Exception as e:
    print(f"🛑 Error: {e}")

In [ ]:
# 1. Volvemos a registrar el correo de Alice en la memoria activa
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Messi <sarah.chen@company.com>",
    "subject": "Consulta rápida sobre la documentación de la API",
    "email_thread": """Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice"""
}

# 2. Configuración de seguridad para evitar bucles (máximo 10 pasos)
configuracion = {"recursion_limit": 10}

try:
    print("🚀 Invocando al sistema con el correo de Alice...\n" + "-"*50)
    
    # Executamos el flujo sobre la app compilada
    response = app.invoke({"email_input": email_input}, config=configuracion)
    
    print("-"*50 + "\n✅ ¡El grafo terminó su ejecución con éxito!")
    
    # 3. Mostramos la respuesta final del agente
    if "messages" in response and len(response["messages"]) > 0:
        print("\n🤖 Respuesta redactada por el agente:\n")
        print(response["messages"][-1].content)
    else:
        print("\n📭 El flujo terminó en el router (Correo ignorado/notificado).")
        
except Exception as e:
    print("-"*50)
    print(f"🛑 Error durante la ejecución del grafo: {e}")

In [ ]:
# 1. RE-DEFINIMOS LOS PROMPTS DEL CURSO (Ajustados con las llaves que descubrimos)
triage_system_prompt = """
Sos un asistente de triaje de correos electrónicos. 
Tu trabajo es clasificar el mail entrante para el usuario {full_name} ({name}).
Antecedentes del usuario: {user_profile_background}

Reglas de triaje:
- Ignorar si coincide con: {triage_ignore}
- Notificar si coincide con: {triage_notify}
- Responder si coincide con: {triage_respond}

Por favor, clasifica el correo electrónico.
"""

triage_user_prompt = """
De: {author}
Para: {to}
Asunto: {subject}
Hilo del correo:
{email_thread}
"""

# 2. RE-DEFINIMOS EL PERFIL Y LAS REGLAS DE TRIAJE
profile = {
    "full_name": "Sarah Messi",
    "name": "Sarah",
    "user_profile_background": "Líder de Desarrollo y Arquitecta de Software encargada de las APIs corporativas."
}

prompt_instructions = {
    "triage_rules": {
        "ignore": "Mails de marketing, promociones, spam o boletines generales.",
        "notify": "Mails informativos de la empresa, alertas de sistemas o actualizaciones de calendario que no requieran acción.",
        "respond": "Consultas técnicas legítimas de compañeros, reportes de bugs en producción o solicitudes directas de asistencia."
    }
}

# 3. VOLVEMOS A CARGAR EL MAIL DE ALICE
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Messi <sarah.messi@company.com>",
    "subject": "Consulta rápida sobre la documentación de la API",
    "email_thread": """Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice"""
}

# 4. EJECUTAMOS EL INVOKE SEGURO
configuracion = {"recursion_limit": 10}

try:
    print("🚀 Invocando al sistema con el entorno 100% restaurado...\n" + "-"*50)
    
    # Ejecución del grafo
    response = app.invoke({"email_input": email_input}, config=configuracion)
    
    print("-"*50 + "\n✅ ¡El grafo terminó su ejecución con éxito!")
    
    # 5. Mostramos la respuesta final
    if "messages" in response and len(response["messages"]) > 0:
        print("\n🤖 Respuesta redactada por el agente:\n")
        print(response["messages"][-1].content)
    else:
        print("\n📭 El flujo terminó en el router (Correo filtrado).")
        
except Exception as e:
    print("-"*50)
    print(f"🛑 Error durante la ejecución del grafo: {e}")

In [ ]:
import os
from typing import Literal
from typing_extensions import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent

print("🔄 Iniciando reconstrucción total del entorno...")

# 1. INICIALIZACIÓN DE MODELOS (Groq Llama 3.3)
# (Asegurate de tener cargada tu API KEY de Groq en las variables de entorno de Colab si es necesario)
try:
    llm_router = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    llm = llm_router  # Usamos el mismo modelo para el agente de respuesta
    print("🤖 Modelos de Groq inicializados correctamente.")
except Exception as e:
    print(f"⚠️ Error al inicializar Groq (revisá tu API KEY): {e}")

# 2. DEFINICIÓN DEL ESTADO (State)
class State(TypedDict):
    email_input: dict
    messages: list

# 3. PROMPTS Y CONFIGURACIONES DEL CURSO
triage_system_prompt = """
Sos un asistente de triaje de correos electrónicos. 
Tu trabajo es clasificar el mail entrante para el usuario {full_name} ({name}).
Antecedentes del usuario: {user_profile_background}

Reglas de triaje:
- Ignorar si coincide con: {triage_ignore}
- Notificar si coincide con: {triage_notify}
- Responder si coincide con: {triage_respond}

Por favor, clasifica el correo electrónico.
"""

triage_user_prompt = """
De: {author}
Para: {to}
Asunto: {subject}
Hilo del correo:
{email_thread}
"""

profile = {
    "full_name": "Sarah Messi",
    "name": "Sarah",
    "user_profile_background": "Líder de Desarrollo y Arquitecta de Software encargada de las APIs corporativas."
}

prompt_instructions = {
    "triage_rules": {
        "ignore": "Mails de marketing, promociones, spam o boletines generales.",
        "notify": "Mails informativos de la empresa, alertas de sistemas o actualizaciones de calendario que no requieran acción.",
        "respond": "Consultas técnicas legítimas de compañeros, reportes de bugs en producción o solicitudes directas de asistencia."
    }
}

# 4. DEFINICIÓN DE LOS NODOS (Funciones)
def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_ignore=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_respond=prompt_instructions["triage_rules"]["respond"],
        examples=None
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    
    # Invocación al modelo clasificador estructurado
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    
    if result.classification == "respond":
        print("📧 Clasificación: RESPONDER - Este e-mail requiere una respuesta")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas")
        update = None
        goto = "__end__"
    elif result.classification == "notify":
        print("🔔 Clasificación: NOTIFICAR - Este e-mail contiene informaciones importantes")
        update = None
        goto = "__end__"
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")
        
    return Command(goto=goto, update=update)

# Re-creamos el agente ReAct de respuesta sin herramientas para evitar loops
agent = create_react_agent(
    model=llm, 
    tools=[],
    prompt="Sos un asistente corporativo experto. Responde de forma clara y profesional al email del usuario."
)
print("🎯 Nodos del grafo preparados.")

# 5. CONSTRUCCIÓN Y COMPILACIÓN DEL GRAFO
email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)
email_agent.add_edge(START, "triage_router")

app = email_agent.compile()
print("📊 Variable 'app' compilada con éxito.")

# 6. ENTRADA DE PRUEBA (Mail de Alice)
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Messi <sarah.messi@company.com>",
    "subject": "Consulta rápida sobre la documentación de la API",
    "email_thread": """Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice"""
}

# 7. INVOCACIÓN SEGURA
print("\n🚀 Invocando al sistema multi-agente...\n" + "-"*50)
try:
    response = app.invoke({"email_input": email_input}, config={"recursion_limit": 10})
    print("-"*50 + "\n✅ ¡El grafo terminó su ejecución con éxito!")
    
    if "messages" in response and len(response["messages"]) > 0:
        print("\n🤖 Respuesta final redactada por el agente:\n")
        print(response["messages"][-1].content)
    else:
        print("\n📭 El flujo terminó en el router (Correo filtrado).")
except Exception as e:
    print(f"🛑 Error durante la ejecución: {e}")

In [ ]:
import os
from typing import Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent

print("🔄 Corrigiendo la estructura de salida del Router...")

# 1. ESQUEMA PYDANTIC PARA FORZAR LA CLASIFICACIÓN
class TriageOutput(BaseModel):
    classification: Literal["respond", "ignore", "notify"] = Field(
        description="Clasificación del correo electrónico según las reglas."
    )

# 2. INICIALIZACIÓN DE MODELOS
llm_base = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Vinculamos el esquema al modelo para que devuelva un objeto con .classification
llm_router = llm_base.with_structured_output(TriageOutput)
llm = llm_base 

# 3. DEFINICIÓN DEL ESTADO
class State(TypedDict):
    email_input: dict
    messages: list

# 4. PROMPTS Y CONFIGURACIONES
triage_system_prompt = """
Sos un asistente de triaje de correos electrónicos. 
Tu trabajo es clasificar el mail entrante para el usuario {full_name} ({name}).
Antecedentes del usuario: {user_profile_background}

Reglas de triaje:
- Ignorar si coincide con: {triage_ignore}
- Notificar si coincide con: {triage_notify}
- Responder si coincide con: {triage_respond}

Por favor, clasifica el correo electrónico seleccionando una de las opciones válidas: 'respond', 'ignore' o 'notify'.
"""

triage_user_prompt = """
De: {author}
Para: {to}
Asunto: {subject}
Hilo del correo:
{email_thread}
"""

profile = {
    "full_name": "Sarah Messi",
    "name": "Sarah",
    "user_profile_background": "Líder de Desarrollo y Arquitecta de Software encargada de las APIs corporativas."
}

prompt_instructions = {
    "triage_rules": {
        "ignore": "Mails de marketing, promociones, spam o boletines generales.",
        "notify": "Mails informativos de la empresa, alertas de sistemas o actualizaciones de calendario que no requieran acción.",
        "respond": "Consultas técnicas legítimas de compañeros, reportes de bugs en producción o solicitudes directas de asistencia."
    }
}

# 5. DEFINICIÓN DEL ROUTER CORREGIDO
def triage_router(state: State) -> Command[Literal["response_agent", "__end__"]]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_ignore=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_respond=prompt_instructions["triage_rules"]["respond"],
        examples=None
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    
    # 'result' ahora sí va a ser un objeto con el atributo '.classification'
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    
    if result.classification == "respond":
        print(f"📧 Clasificación: RESPONDER ({result.classification}) - Este e-mail requiere una respuesta.")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print(f"🚫 Clasificación: IGNORAR ({result.classification}) - Este e-mail puede ser ignorado sin problemas.")
        update = None
        goto = "__end__"
    elif result.classification == "notify":
        print(f"🔔 Clasificación: NOTIFICAR ({result.classification}) - Este e-mail contiene informaciones importantes.")
        update = None
        goto = "__end__"
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")
        
    return Command(goto=goto, update=update)

# Re-creamos el agente ReAct de respuesta sin herramientas
agent = create_react_agent(
    model=llm, 
    tools=[],
    prompt="Sos un asistente corporativo experto. Responde de forma clara y profesional al email del usuario."
)

# 6. CONSTRUCCIÓN DEL GRAFO
email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", agent)
email_agent.add_edge(START, "triage_router")

app = email_agent.compile()
print("📊 Variable 'app' compilada con salida estructurada habilitada.")

# 7. ENTRADA DE PRUEBA (Mail de Alice)
email_input = {
    "author": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Messi <sarah.messi@company.com>",
    "subject": "Consulta rápida sobre la documentación de la API",
    "email_thread": """Hola Sarah,

Estaba revisando la documentación de la API para el nuevo servicio de autenticación y noté que algunos endpoints parecen faltar en las especificaciones. ¿Podrías ayudarme a aclarar si esto fue intencional o si debemos actualizar la documentación?

Específicamente, estoy buscando:
- /auth/refresh
- /auth/validate

¡Gracias!
Alice"""
}

# 8. EJECUCIÓN
print("\n🚀 Invocando al sistema multi-agente con salida estructurada...\n" + "-"*50)
try:
    response = app.invoke({"email_input": email_input}, config={"recursion_limit": 10})
    print("-"*50 + "\n✅ ¡El grafo terminó su ejecución con éxito!\n")
    
    # Recorremos e imprimimos de forma estética todo el historial de mensajes del grafo
    if "messages" in response and len(response["messages"]) > 0:
        for m in response["messages"]:
            m.pretty_print()
            print("-" * 30) # Separador visual opcional entre mensajes
    else:
        print("📭 El flujo terminó en el router (Correo filtrado).")
        
except Exception as e:
    print(f"🛑 Error durante la ejecución: {e}")